# MobileNetV2: ajuste, piloto y ejecución definitiva

Este notebook entrena y evalúa MobileNetV2 con las funciones de `src/` (`train.py`,
`tune.py`, `pilot.py`, `results.py`). Todos los parámetros se leen de `configs/base.yaml`
(sección `train`) y `configs/mobilenetv2.yaml`. El ajuste y el piloto usan solo los pliegues
de entrenamiento; ninguna celda evalúa test.

## 1. Configuración

Modelo y flags de reejecución. Con `RERUN_AJUSTE` o `RERUN_PILOTO` en `False`, las secciones 3
y 4 cargan los resultados guardados si existen; con `True`, llaman a `src.tune` o `src.pilot`,
que reutilizan las corridas ya terminadas y ejecutan las pendientes.

In [ ]:
MODELO = "mobilenetv2"
RERUN_AJUSTE = False
RERUN_PILOTO = False

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src import data, results
from src.pilot import run_pilot
from src.train import build_model, configure_runtime, load_model_config
from src.tune import run_tuning
from src.utils import environment_info, set_seed

Carga de las configuraciones, memory growth en la GPU, política de precisión
(`train.mixed_precision`) y semilla (`seed` de `base.yaml`).

In [ ]:
cfg = data.load_config()
cfg_modelo = load_model_config(MODELO)
configure_runtime(cfg)
set_seed(cfg["seed"], deterministic=cfg["train"]["deterministic"])
rutas = cfg["paths"]
cfg_train = cfg["train"]

pd.Series({
    "modelo": cfg_modelo["name"],
    "aplicación de Keras": cfg_modelo["keras_application"],
    "pesos": cfg_modelo["weights"],
    "entrada": f"{cfg_modelo['input_size']} x {cfg_modelo['input_size']}",
    "clases": ", ".join(data.class_names(cfg)),
    "batch_size": cfg_train["batch_size"],
    "mixed_precision": cfg_train["mixed_precision"],
    "deterministic": cfg_train["deterministic"],
    "épocas fase A": cfg_train["warmup"]["epochs"],
    "épocas máximas fase B": cfg_train["finetune"]["max_epochs"],
    "semilla del notebook": cfg["seed"],
}, name="valor").to_frame()

GPU detectada y versiones de Python, TensorFlow, Keras, CUDA y cuDNN (estas dos, las de
compilación de TensorFlow).

In [ ]:
results.environment_table(environment_info())

## 2. Modelo

Arquitectura: entrada en [0, 255], `preprocess_input` de la aplicación, backbone con
`training=False`, `GlobalAveragePooling2D`, `Dropout` y `Dense` softmax en float32. El dropout
es el del YAML, o 0.0 si aún es null (no cambia el número de parámetros).

In [ ]:
modelo = build_model(cfg, MODELO)
modelo.summary()

Parámetros entrenables y no entrenables con el backbone congelado (fase A) y descongelado
(fase B). Los no entrenables restantes en la fase B son las medias y varianzas móviles de
BatchNormalization.

In [ ]:
results.model_parameter_table(modelo)

## 3. Ajuste de hiperparámetros

Búsqueda aleatoria de `train.tuning`: `n_trials` configuraciones muestreadas con
`sampler_seed`, una corrida por configuración con `seed` y `val_fold` fijos y la fase B
limitada a `tuning.max_epochs`.

In [ ]:
pd.DataFrame(cfg_train["tuning"]["space"]).T

### 3.1 Ejecución

Carga `outputs/tuning/<modelo>/trials.csv` si existe y `RERUN_AJUSTE` es `False`; si no, llama
a `run_tuning`. Si el ajuste se interrumpió, el archivo contiene solo los ensayos terminados.

In [ ]:
dir_ajuste = rutas["tuning_dir"] / MODELO
ensayos, origen = results.load_or_run(dir_ajuste / "trials.csv", lambda: run_tuning(cfg, MODELO), RERUN_AJUSTE)
pd.Series({"archivo": str(dir_ajuste / "trials.csv"), "estado": origen, "ensayos": len(ensayos)},
          name="valor").to_frame()

### 3.2 Resultados

Configuraciones ordenadas por exactitud balanceada de validación (`best_epoch` es la época de
la fase B cuyos pesos se restauraron).

In [ ]:
results.tuning_table(ensayos)

Mejor configuración según `best.json`.

In [ ]:
mejor = data.load_json(dir_ajuste / "best.json")
pd.Series({"ensayo": mejor["trial"], **mejor["hyperparameters"],
           "val_balanced_accuracy": mejor["val_balanced_accuracy"], "best_epoch": mejor["best_epoch"]},
          name="valor").to_frame()

Exactitud balanceada de validación frente a cada hiperparámetro (eje logarítmico para los
de distribución loguniforme). El número junto a cada punto es el índice del ensayo; el círculo
rojo marca el mejor.

In [ ]:
results.plot_metric_vs_hparams(ensayos, cfg_train["tuning"]["space"], best_trial=mejor["trial"])
plt.show()

### 3.3 Hiperparámetros seleccionados

Valores de la sección `hyperparameters` de `configs/mobilenetv2.yaml` frente a `best.json`. Se
emite un aviso si alguno es null o difiere.

In [ ]:
results.compare_hparams(cfg_modelo, mejor)

## 4. Estudio piloto

Corridas de `train.pilot.pairs` (semilla, pliegue de validación) con los hiperparámetros del
YAML, sin test.

### 4.1 Ejecución

Carga `outputs/pilot/<modelo>/summary.csv` si existe y `RERUN_PILOTO` es `False`; si no, llama
a `run_pilot`.

In [ ]:
dir_piloto = rutas["pilot_dir"] / MODELO
piloto, origen = results.load_or_run(dir_piloto / "summary.csv", lambda: run_pilot(cfg, MODELO), RERUN_PILOTO)
corridas = results.run_dirs_from_table(piloto)
pd.Series({"archivo": str(dir_piloto / "summary.csv"), "estado": origen, "corridas": len(corridas)},
          name="valor").to_frame()

### 4.2 Resultados

Exactitud balanceada de validación, épocas y tiempo de entrenamiento por corrida. Las filas
`mean`, `std` (ddof = 1) y `cv` (std / mean) se refieren a la exactitud balanceada.

In [ ]:
piloto[["row", "seed", "val_fold", "val_balanced_accuracy", "best_epoch", "epochs_warmup",
        "epochs_finetune", "train_time_s"]]

### 4.3 Curvas de entrenamiento

Pérdida y exactitud balanceada de train (discontinua) y validación (continua) por época global.
La línea gris marca el paso de la fase A a la B y la roja punteada la época restaurada. La
pérdida de train está ponderada por los pesos de clase; la de validación no. Las métricas de
train se calculan sobre imágenes aumentadas y con dropout activo.

In [ ]:
historias = results.load_histories(corridas)
results.plot_training_curves(historias)
plt.show()

### 4.4 Métricas de validación por corrida

Métricas globales de validación de cada corrida con los pesos restaurados, y su media y
desviación estándar (ddof = 1).

In [ ]:
results.add_mean_std(results.run_metrics_table(corridas, split="val"))

Precisión, sensibilidad y F1 por clase: media y desviación estándar entre corridas, y valores
de cada corrida.

In [ ]:
por_clase = results.per_class_table(corridas, split="val")
display(results.per_class_summary(por_clase))
por_clase

Matriz de confusión de validación acumulada sobre las corridas (suma de las matrices) y
normalizada por fila.

In [ ]:
cm, etiquetas = results.pooled_confusion(corridas, split="val")
results.plot_confusion_matrix(cm, etiquetas, normalize=True,
                              title=f"Validación, {len(corridas)} corridas acumuladas (normalizada por fila)")
plt.show()

### 4.5 Costo computacional

Tiempo de entrenamiento por fase, épocas, tiempo por época de la fase B, inferencia por imagen
en validación (lotes de `batch_size`, tras una predicción de calentamiento) y parámetros.

In [ ]:
results.add_mean_std(results.cost_table(corridas))

GPU y versiones registradas en la primera corrida del piloto.

In [ ]:
results.environment_table(results.load_run_info(corridas[0])["environment"])

## 5. Ejecución definitiva

Esta sección se completa después del preregistro.